In [1]:
import argparse
import time
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Filepaths
stop_times_path = 'Datasets/stop_times.txt'
stops_path = 'Datasets/stops.txt'
trips_path = 'Datasets/trips.txt'
calendar_path = 'Datasets/calendar.txt'
api_path = 'Datasets/api_mar1.csv'

In [3]:
def process_gtfs_time(time_str):
    '''
    This function gets the time string from GTFS dataset and turns it into minutes since midnight
    '''

    try:
        h, m, s = map(int, str(time_str).split(':'))
        return h*60 + m + s/60
    except Exception:
        return np.nan
    
def process_api_time(df):
    dt = pd.to_datetime(df)
    return dt.dt.hour * 60 + dt.dt.minute + dt.dt.second/60

In [4]:
def preprocess_static_data(stop_times_path, stops_path, trips_path, calendar_path, cache_dir='gtfs_cache'):
    '''
    This function merges stop_times, trips, stops, and their calendars in a Parquet file
    *** OFFLINE STEP
    '''

    print('Pre-processing GTFS data')

    # Setting the directories
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_path = cache_dir / 'static_data.parquet'
    cal_path = cache_dir / 'trips_calendar.parquet'

    # Initializing clock for performance analysis
    t0 = time.time()

    # Reading trips and stop files
    trips = pd.read_csv(trips_path, dtype={
        'route_id': str,
        'trip_id': str,
        'shape_id': str,
        'service_id': str
        })
    
    stops = pd.read_csv(stops_path, dtype={
        'stop_id': str
    })

    # Building the merged dataset (using chunks because of large size)
    trip_id_set = set(trips['trip_id'].astype(str))
    chunks = []
    for chunk in pd.read_csv(stop_times_path, 
                             dtype={'trip_id': str, 'stop_id': str},
                             chunksize=500_000):
        chunks.append(chunk[chunk['trip_id'].isin(trip_id_set)])
    
    stop_times = pd.concat(chunks, ignore_index=True)

    # Adjusting data types
    stop_times['arrival_minutes'] = stop_times['arrival_time'].apply(process_gtfs_time)
    stop_times['departure_minutes'] = stop_times['departure_time'].apply(process_gtfs_time)
    stop_times['shape_dist_traveled'] = pd.to_numeric(stop_times['shape_dist_traveled'], errors='coerce')
    stop_times = stop_times.dropna(subset=["shape_dist_traveled", "arrival_minutes"])
    stop_times = stop_times.sort_values(["trip_id", "stop_sequence"]).reset_index(drop=True)

    # Merging datasets
    stop_times = stop_times.merge(
        stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
        on = 'stop_id', how = 'left'
    )

    stop_times = stop_times.merge(
        trips[['trip_id', 'route_id', 'shape_id', 'direction', 'schd_trip_id', 'service_id']],
        on='trip_id', how='left')
    
    stop_times['route_id'] = stop_times['route_id'].astype(str).str.strip()
    important_cols = ['route_id', 'trip_id', 'shape_id', 'stop_sequence', 'arrival_minutes', 'departure_minutes',
                      'shape_dist_traveled', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'service_id', 
                      'direction', 'schd_trip_id']
    
    stop_times = stop_times[[c for c in important_cols if c in stop_times.columns]]
    stop_times = stop_times.sort_values(['route_id', 'shape_id', 'trip_id', 'stop_sequence'])

    # Building the calendar lookup for trips
    calendar = pd.read_csv(calendar_path, dtype=str)
    day_cols = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
    for col in day_cols:
        calendar[col] = calendar[col].astype(int)
    trips_calendar = trips[['route_id', 'service_id', 'trip_id', 'shape_id', 'schd_trip_id']].merge(
        calendar[['service_id'] + day_cols], on = 'service_id', how = 'left'
    )

    # Writing Parquet tables
    pq.write_table(
        pa.Table.from_pandas(stop_times, preserve_index=False),
        out_path, row_group_size=100_000, compression='snappy'
    )
    
    pq.write_table(
        pa.Table.from_pandas(trips_calendar, preserve_index=False),
        cal_path, compression='snappy'
        )
    
    print('GTFS pre-processed')
    

In [5]:
def load_gtfs_cache(cache_dir='gtfs_cache'):
    '''
    This function reads the pre-processed gtfs parquet files
    '''
    print('Loading GTFS pre-processed parquet files')

    cache_dir = Path(cache_dir)
    out_path = cache_dir / 'static_data.parquet'
    cal_path = cache_dir / 'trips_calendar.parquet'

    if not out_path.exists():
        raise FileNotFoundError('GTFS cache not found — rerun the pre-processing')
    
    t0 = time.time()
    con = duckdb.connect()
    stop_times = con.execute(f"SELECT * FROM read_parquet('{out_path}')").df()
    trips_calendar = con.execute(f"SELECT * FROM read_parquet('{cal_path}')").df()

    con.close()

    print('GTFS loaded')

    return stop_times, trips_calendar

In [6]:
def build_trip_index(stop_times, trips_calendar):
    '''
    Converts GTFS stop_times into fast lookup structures for matching.

    Returns:
        trip_store:  {trip_id: dict}          — full trip data as numpy arrays
        shape_index: {shape_id: [trip_id, …]} — all trips per shape, for fallback matching
    '''
    print('Building trip index...')
    t0 = time.time()

    # Build service_id → active weekday set {0=Mon … 6=Sun}
    day_cols = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
    svc_days = {}
    trip_svc = {}
    if all(c in trips_calendar.columns for c in day_cols):
        for _, row in trips_calendar.iterrows():
            sid    = str(row['service_id'])
            active = {i for i, col in enumerate(day_cols) if int(row[col]) == 1}
            svc_days[sid] = active
        trip_svc = dict(zip(
            trips_calendar['trip_id'].astype(str),
            trips_calendar['service_id'].astype(str)
        ))

    trip_store  = {}  # trip_id → arrays
    shape_index = {}  # shape_id → [trip_id, ...]

    for (route_id, trip_id, shape_id), grp in stop_times.groupby(['route_id', 'trip_id', 'shape_id']):
        route_id = str(route_id).strip()
        trip_id  = str(trip_id)
        shape_id = str(shape_id)
        grp      = grp.sort_values('stop_sequence')
        svc_id   = trip_svc.get(trip_id)

        trip_store[trip_id] = {
            'trip_id':         trip_id,
            'route_id':        route_id,
            'shape_id':        shape_id,
            'service_id':      svc_id,
            'active_days':     svc_days.get(svc_id, set(range(7))),
            'dists':           grp['shape_dist_traveled'].values.astype(np.float32),
            'arr_min':         grp['arrival_minutes'].values.astype(np.float32),
            'dep_min':         grp['departure_minutes'].values.astype(np.float32),
            'stop_ids':        grp['stop_id'].values,
            'stop_names':      grp['stop_name'].values,
            'stop_lats':       grp['stop_lat'].values.astype(np.float32),
            'stop_lons':       grp['stop_lon'].values.astype(np.float32),
            # First-stop anchor
            'first_stop_name': grp['stop_name'].values[0],
            'first_dep_min':   float(grp['departure_minutes'].values[0]),
            'first_dep_sec':   int(round(float(grp['departure_minutes'].values[0]) * 60)),
        }

        if shape_id not in shape_index:
            shape_index[shape_id] = []
        shape_index[shape_id].append(trip_id)

    print(f'Trip index built: {len(trip_store):,} trips | {len(shape_index):,} shapes | {time.time()-t0:.1f}s')
    return trip_store, shape_index

In [7]:
def build_stst_index(trip_store, shape_index, unique_pids):
    '''
    Builds (route_id, pid_str, dep_seconds) → trip_id lookup.

    route_id is included because the same pid suffix can appear on multiple
    routes, and two routes can share a first-stop departure time.
    dep_seconds matches the raw stst column in the vehicle feed (int, seconds
    past midnight) so the lookup is a direct dict get() with no conversion.

    Called after load_api_data() so actual vehicle pid values are known.
    '''
    print('Building stst_index...')
    t0 = time.time()

    stst_index = {}
    for pid_str in unique_pids:
        pid_str = str(pid_str).strip()
        if pid_str in ('', 'nan', 'None'):
            continue
        for shape_id, trip_ids in shape_index.items():
            if str(shape_id).endswith(pid_str):
                for trip_id in trip_ids:
                    trip = trip_store[trip_id]
                    key  = (trip['route_id'], pid_str, trip['first_dep_sec'])
                    stst_index[key] = trip_id

    print(f'stst_index built: {len(stst_index):,} keys | {len(unique_pids):,} unique pids | {time.time()-t0:.1f}s')
    return stst_index

In [8]:
def load_api_data(api_path):
    '''This function loads the API data using DuckDB'''

    print('Loading API data')
    t0 = time.time()

    path = Path(api_path)
    chunks = []
    for chunk in pd.read_csv(path, 
                             dtype=str,
                             chunksize=500_000):
        chunks.append(chunk)
    
    df = pd.concat(chunks, ignore_index=True)

    # Adjusting data types
    df['vid'] = df['vid'].astype(str).str.strip()
    df['tmstmp'] = pd.to_datetime(df['tmstmp'], errors='coerce')
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
    df['pid'] = df['pid'].astype(str).str.strip()
    df['dly'] = df['dly'].astype(str).str.lower().map({'true':True, 'false':False}).fillna(False)
    df['rt'] = df['rt'].str.strip()
    df['pdist'] = pd.to_numeric(df['pdist'], errors='coerce')
    df['tatripid'] = pd.to_numeric(df['tatripid'], errors='coerce')
    df['origtatripno'] = pd.to_numeric(df['origtatripno'], errors='coerce')
    df['stst'] = pd.to_numeric(df['stst'], errors='coerce')
    df['time_minutes'] = process_api_time(df['tmstmp'])

    df = df.rename(columns={
        'vid': 'vehicle_id',
        'rt': 'route_id'})

    df.head()

    orig = len(df)
    # Dropping rows with no timestamp or location
    df = df.dropna(subset=['tmstmp', 'pdist', 'lat', 'lon']).reset_index(drop=True)
    
    before = len(df)

    before = len(df)
    df = df[df['time_minutes'] > 0].reset_index(drop=True)
    dropped = before - len(df)
    if dropped:
        print(f'Dropped {dropped:,} rows with malformed timestamps (time_minutes=0)')

    # Dropping duplicate observations
    df = df.drop_duplicates(subset=['tmstmp', 'vehicle_id'], keep='first').reset_index(drop=True)

    final = len(df)
    
    print(f'Original dataset length: {orig}\n')
    print(f'Null timestamp or location values: {before-orig}; filtered length: {before}\n')
    print(f'Duplicate observations: {before-final}; filtered length: {final}\n')

    return df

In [9]:
stop_times, trips_calendar = load_gtfs_cache()
api = load_api_data(api_path)

Loading GTFS pre-processed parquet files
GTFS loaded
Loading API data
Dropped 1,577 rows with malformed timestamps (time_minutes=0)
Original dataset length: 4994350

Null timestamp or location values: 0; filtered length: 4994350

Duplicate observations: 24728; filtered length: 4969622



In [10]:
trip_store, shape_index = build_trip_index(stop_times, trips_calendar)

unique_pids = api['pid'].dropna().unique()
stst_index  = build_stst_index(trip_store, shape_index, unique_pids)

# Lookup at match time:
# key = (row['route_id'], str(row['pid']), int(row['stst']))
# trip_id = stst_index.get(key)  → exact trip or None

Building trip index...
Trip index built: 49,936 trips | 782 shapes | 7.3s
Building stst_index...
stst_index built: 45,407 keys | 756 unique pids | 0.1s


In [27]:
def _get_trip_date(tmstmp, stst_sec, rollback_hour=4):
    '''
    Returns the calendar date a trip belongs to.
    
    If the observation is in the early morning (before rollback_hour)
    and stst is in the late evening (>= 20:00), the trip started the
    previous calendar day — roll back the date by one day.
    
    e.g. a trip with stst=82800 (23:00) observed at 00:30 the next day
    belongs to the previous day's schedule.
    '''
    obs_date = tmstmp.date()
    if tmstmp.hour < rollback_hour and stst_sec >= 20 * 3600:
        obs_date = (tmstmp - pd.Timedelta(days=1)).date()
    return obs_date


def _find_bracket(dists, pdist):
    '''
    Find stop bracket index b where dists[b] <= pdist < dists[b+1].
    Returns None if pdist is outside the trip range.
    '''
    b = int(np.searchsorted(dists, pdist, side='right') - 1)
    if b < 0 or b >= len(dists) - 1:
        return None
    return b


def _interpolate_scheduled_time(trip, b, pdist):
    '''
    Linearly interpolate the scheduled time at the vehicle's exact pdist
    between stop b (departure) and stop b+1 (arrival).
    '''
    dists      = trip['dists']
    dist_range = float(dists[b + 1] - dists[b])
    if dist_range <= 0:
        return float(trip['dep_min'][b])
    frac = (pdist - float(dists[b])) / dist_range
    return float(trip['dep_min'][b]) + frac * (float(trip['arr_min'][b + 1]) - float(trip['dep_min'][b]))


def _compute_delay(row, trip):
    b = _find_bracket(trip['dists'], row['pdist'])

    if b is None:
        return {
            'delay_minutes':             None,
            'schedule_elapsed_minutes':  None,
            'first_stop_name':           trip['first_stop_name'],
            'scheduled_departure_stop1': trip['first_dep_min'],
            'prev_stop_name':            None,
            'next_stop_name':            None,
        }

    interp_time = _interpolate_scheduled_time(trip, b, row['pdist'])

    # Overnight correction: if this observation's trip_date was rolled back
    # (i.e. the bus started the previous calendar day), the wall-clock
    # time_minutes is small (00:xx) while the scheduled time is large (23:xx).
    # Add 1440 to bring the observation into the same reference frame as GTFS.
    obs_time = row['time_minutes']
    if row.get('_overnight', False):
        obs_time += 1440

    schedule_elapsed = interp_time - trip['first_dep_min']
    delay            = obs_time - interp_time

    return {
        'delay_minutes':             round(delay, 1),
        'schedule_elapsed_minutes':  round(schedule_elapsed, 1),
        'first_stop_name':           trip['first_stop_name'],
        'scheduled_departure_stop1': trip['first_dep_min'],
        'prev_stop_name':            trip['stop_names'][b],
        'next_stop_name':            trip['stop_names'][b + 1],
    }


def _score_candidates(candidates, trip_store, pdist, obs_time, max_score_minutes=45):
    '''
    Fallback: pick the trip whose interpolated scheduled time at the
    vehicle's current pdist is closest to the observed time.
    Returns best trip_id only if score is within max_score_minutes.
    Returns None if no candidate fits within the threshold.
    '''
    best_trip_id = None
    best_score   = np.inf

    for trip_id in candidates:
        trip = trip_store[trip_id]
        b    = _find_bracket(trip['dists'], pdist)
        if b is None:
            continue
        score = abs(obs_time - _interpolate_scheduled_time(trip, b, pdist))
        if score < best_score:
            best_score   = score
            best_trip_id = trip_id

    return best_trip_id if best_score <= max_score_minutes else None


def match_trips_bus(api, trip_store, shape_index, stst_index):
    '''
    Matches each vehicle observation to a scheduled GTFS trip and computes delay.

    Grouping: (vehicle_id, route_id, pid, stst, trip_date) — each unique
    combination is one trip run. trip_date rolls back by one day for overnight
    trips observed past midnight.

    Lookup hierarchy:
      1. stst_index[(route_id, pid, stst)] → exact trip_id, O(1)
         — rejected if vehicle's median time is >50 min from trip's first_dep_min
      2. shape_index[shape_id] filtered by route + weekday → score-based
      3. route_index[route_id] filtered by weekday → broader score-based
      4. no candidates → unscheduled
    '''
    print('Matching vehicles to trips...')
    t0 = time.time()

    api = api.copy()
    api['trip_date'] = [
        _get_trip_date(ts, st)
        for ts, st in zip(api['tmstmp'], api['stst'])
    ]
    api['_overnight'] = api['trip_date'] != api['tmstmp'].dt.date
    api['weekday']    = api['tmstmp'].dt.weekday

    # Build route → [trip_id, ...] for fallback
    route_index = {}
    for trip_id, trip in trip_store.items():
        route_index.setdefault(trip['route_id'], []).append(trip_id)

    results     = []
    group_cols  = ['vehicle_id', 'route_id', 'pid', 'stst', 'trip_date']

    for group_keys, grp in api.groupby(group_cols, dropna=False):
        vehicle_id, route_id, pid_str, stst_sec, trip_date = group_keys
        weekday  = int(grp['weekday'].iloc[0])
        pid_str  = str(pid_str).strip()
        stst_int = int(stst_sec) if pd.notna(stst_sec) else None
        is_overnight = grp['_overnight'].any()

        obs_time_median = grp['time_minutes'].median()
        if is_overnight:
            obs_time_median += 1440

        # ── 1. stst exact lookup ──────────────────────────────────────────
        trip_id    = None
        match_type = 'unscheduled'

        if stst_int is not None:
            candidate = stst_index.get((route_id, pid_str, stst_int))
            if candidate:
                trip = trip_store[candidate]
                if abs(obs_time_median - trip['first_dep_min']) <= 50:
                    trip_id    = candidate
                    match_type = 'stst'
                else:
                    match_type = 'stst_rejected'

        # ── 2. shape fallback ─────────────────────────────────────────────
        if trip_id is None:
            candidates = [
                t for shape_id, trip_ids in shape_index.items()
                if str(shape_id).endswith(pid_str)
                for t in trip_ids
                if trip_store[t]['route_id'] == route_id
                and weekday in trip_store[t]['active_days']
            ]

            # ── 3. full route fallback ────────────────────────────────────
            if not candidates:
                candidates = [
                    t for t in route_index.get(route_id, [])
                    if weekday in trip_store[t]['active_days']
                ]

            if candidates:
                trip_id = _score_candidates(candidates, trip_store,
                                            grp['pdist'].median(),
                                            obs_time_median)
                if trip_id:
                    match_type = 'fallback'
                else:
                    match_type = 'unscheduled'

        # ── 4. compute delay per observation ──────────────────────────────
        trip = trip_store.get(trip_id) if trip_id else None
        for idx, row in grp.iterrows():
            result = _compute_delay(row, trip) if trip else {
                'delay_minutes':             None,
                'schedule_elapsed_minutes':  None,
                'first_stop_name':           None,
                'scheduled_departure_stop1': None,
                'prev_stop_name':            None,
                'next_stop_name':            None,
            }
            results.append({**row, **result,
                            'matched_trip_id': trip_id,
                            'match_type':      match_type})

    result_df = pd.DataFrame(results).reset_index(drop=True)

    n_total         = len(result_df)
    n_stst          = (result_df['match_type'] == 'stst').sum()
    n_stst_rejected = (result_df['match_type'] == 'stst_rejected').sum()
    n_fallback      = (result_df['match_type'] == 'fallback').sum()
    n_unscheduled   = (result_df['match_type'] == 'unscheduled').sum()
    print(f'Total: {n_total:,} | stst: {n_stst:,} ({n_stst/n_total*100:.1f}%) | '
          f'stst_rejected: {n_stst_rejected:,} ({n_stst_rejected/n_total*100:.1f}%) | '
          f'fallback: {n_fallback:,} ({n_fallback/n_total*100:.1f}%) | '
          f'unscheduled: {n_unscheduled:,} ({n_unscheduled/n_total*100:.1f}%)')
    print(f'Done in {time.time()-t0:.1f}s')

    return result_df

In [36]:
result = match_trips_bus(api, trip_store, shape_index, stst_index)

Matching vehicles to trips...
Total: 4,969,622 | stst: 4,116,743 (82.8%) | stst_rejected: 0 (0.0%) | fallback: 732,952 (14.7%) | unscheduled: 119,927 (2.4%)
Done in 268.2s


In [37]:
# Full dataset analysis
print('=== MATCH TYPE BREAKDOWN ===')
print(result['match_type'].value_counts())

print('\n=== DELAY DISTRIBUTION (stst-matched) ===')
stst = result[result['match_type'] == 'stst']
print(stst['delay_minutes'].describe())

print('\n=== DELAY DISTRIBUTION (fallback) ===')
fb = result[result['match_type'] == 'fallback']
print(fb['delay_minutes'].describe())

print('\n=== UNSCHEDULED BREAKDOWN BY ROUTE ===')
unsch = result[result['match_type'] == 'unscheduled']
route_unsch = (unsch.groupby('route_id')
               .size()
               .sort_values(ascending=False)
               .head(15))
print(route_unsch)

print('\n=== WORST DELAY OUTLIERS (stst-matched) ===')
print(stst.nsmallest(5, 'delay_minutes')[
    ['vehicle_id','tmstmp','route_id','delay_minutes',
     'first_stop_name','prev_stop_name']
].to_string())
print()
print(stst.nlargest(5, 'delay_minutes')[
    ['vehicle_id','tmstmp','route_id','delay_minutes',
     'first_stop_name','prev_stop_name']
].to_string())

=== MATCH TYPE BREAKDOWN ===
match_type
stst           4116743
fallback        732952
unscheduled     119927
Name: count, dtype: int64

=== DELAY DISTRIBUTION (stst-matched) ===
count    4.079061e+06
mean     7.513535e-01
std      5.124216e+00
min     -1.071000e+02
25%     -1.900000e+00
50%      0.000000e+00
75%      2.600000e+00
max      3.155000e+02
Name: delay_minutes, dtype: float64

=== DELAY DISTRIBUTION (fallback) ===
count    710618.000000
mean          1.479106
std           8.287610
min         -78.500000
25%          -2.200000
50%           0.200000
75%           3.400000
max         345.100000
Name: delay_minutes, dtype: float64

=== UNSCHEDULED BREAKDOWN BY ROUTE ===
route_id
9      8175
53     6979
79     6883
62     6753
22     6273
66     5892
49     5700
4      5620
77     4504
20     3939
63     3825
60     3585
151    3456
8      3267
N5     2944
dtype: int64

=== WORST DELAY OUTLIERS (stst-matched) ===
        vehicle_id              tmstmp route_id  delay_minutes  

In [ ]:
def match_trips_bus(stop_times, trips_calendar, api):
    '''This function matches buses to trips'''

    # 1. build the indexing via route id, pid, and day of the week
    # 2. finds the first stop for a given vehicle in that trip
    # 3. tries to match the time it left the first stop to the time for a scheduled trip
        # 3b. tests whether the timing it advances corresponds to the route itself
    # 4. if there isn't a first stop match, then check through the trips associated to that route that haven't been matched
    # 5. if there isn't a matching trip to that bus, flag it as unscheduled
    # 6. if there isn't a matching bus to the trip, flag bus as ghost.

    ### OBS: Since we're doing a temporal analysis, this affects every observation for that bus 
    #        until it finishes the specific trip it's completing

In [ ]:
def identify_ghost_buses(api):
    '''
    This function identifies ghost buses
    
    We identify ghost buses in three ways:
    1. Sustained freeze: bus does not move for 20+ minutes
    2. Outside of route: if bus goes off route by more than 2km
    3. Impossible distance jump (it skipped a stop)
    '''